In [26]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
import plotly.express as px

In [43]:
inc_trunc = pd.read_csv('data/incident_truncated.csv')
mob_trunc = pd.read_csv('data/mobilisation_truncated.csv')

/var/folders/ml/mylzx0q15zj5ydny113g9v_80000gn/T/ipykernel_38999/2747966397.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  mob_trunc = pd.read_csv('data/mobilisation_truncated.csv')


In [ ]:

#mob_3 = pd.read_csv('data/Mobilisation_data_2021_2024.csv')
#mob_4 = pd.read_csv('data/Mobilisation_data_2025.csv')

#inc_2 = pd.read_csv('data/Incident_data_2018_2023.csv')
#inc_3 = pd.read_csv('data/Incident_data_2024_onwards.csv')

#mob_data = pd.concat([mob_3, mob_4], ignore_index=True)
#inc_data_temp = pd.concat([inc_2, inc_3], ignore_index=True)
#inc_data = inc_data_temp[inc_data_temp['CalYear']>=2021]

#save trunced csvs

#inc_data.to_csv('data/Incident_data_combined_2021_2026.csv', index=False)
#mob_data.to_csv('data/Mobilisation_data_combined_2021_2026.csv', index=False)




/var/folders/ml/mylzx0q15zj5ydny113g9v_80000gn/T/ipykernel_38999/2393440497.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  mob_4 = pd.read_csv('data/Mobilisation_data_2025.csv')


In [36]:
inc_all = pd.read_csv('data/Incident_data_combined_2021_2026.csv')
mob_all = pd.read_csv('data/Mobilisation_data_combined_2021_2026.csv')

/var/folders/ml/mylzx0q15zj5ydny113g9v_80000gn/T/ipykernel_38999/1627339509.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  mob_all = pd.read_csv('data/Mobilisation_data_combined_2021_2026.csv')


In [ ]:
missing_data_mob = mob_trunc.isna().mean() *100
missing_data_mob

In [ ]:
missing_data_inc = inc_trunc.isna().mean() *100
missing_data_inc

In [ ]:
pd.set_option('display.max_columns', None)


In [44]:
#Drops duplicates out of Mobilisation file, incident file has no duplicates
mob_trunc.duplicated().sum()
mob_trunc = mob_trunc.drop_duplicates()


In [45]:
#deleate columns in incident file

cols_to_drop_inc = [
    'TimeOfCall', 'StopCodeDescription', 'AddressQualifier', 'Postcode_full', 
    'Postcode_district', 'UPRN', 'USRN', 'IncGeo_BoroughCode', 'ProperCase', 
    'IncGeo_WardCode', 'IncGeo_WardName', 'IncGeo_WardNameNew', 'Latitude', 
    'Longitude', 'FRS', 'IncidentStationGround', 'FirstPumpArriving_AttendanceTime', 
    'FirstPumpArriving_DeployedFromStation', 'SecondPumpArriving_AttendanceTime', 
    'SecondPumpArriving_DeployedFromStation', 'NumStationsWithPumpsAttending', 
    'NumPumpsAttending', 'PumpCount', 'PumpMinutesRounded', 'Notional Cost (£)'
]

inc_trunc = inc_trunc.drop(columns=cols_to_drop_inc)
print(inc_trunc.columns)

Index(['IncidentNumber', 'DateOfCall', 'CalYear', 'HourOfCall',
       'IncidentGroup', 'SpecialServiceType', 'PropertyCategory',
       'PropertyType', 'IncGeo_BoroughName', 'Easting_m', 'Northing_m',
       'Easting_rounded', 'Northing_rounded', 'NumCalls', 'source_file'],
      dtype='object')


In [47]:
#deleate columns in mobilisation file

cols_to_drop_mob = [
    'ResourceMobilisationId', 'Resource_Code', 'PerformanceReporting', 
    'DateAndTimeMobile', 'DateAndTimeArrived', 'TurnoutTimeSeconds', 
    'TravelTimeSeconds', 'DateAndTimeLeft', 'DateAndTimeReturned', 
    'DeployedFromStation_Code', 'DeployedFromLocation', 'PlusCode_Code', 
    'PlusCode_Description', 'DelayCode_Description', 'BoroughName', 'WardName'
]

mob_trunc = mob_trunc.drop(columns=cols_to_drop_mob)


print(mob_trunc.columns)


Index(['IncidentNumber', 'CalYear', 'HourOfCall', 'DateAndTimeMobilised',
       'AttendanceTimeSeconds', 'DeployedFromStation_Name', 'PumpOrder',
       'DelayCodeId', 'source_file'],
      dtype='object')


In [48]:
inc_trunc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 654729 entries, 0 to 654728
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   IncidentNumber      654729 non-null  object 
 1   DateOfCall          654729 non-null  object 
 2   CalYear             654729 non-null  int64  
 3   HourOfCall          654729 non-null  int64  
 4   IncidentGroup       654675 non-null  object 
 5   SpecialServiceType  254128 non-null  object 
 6   PropertyCategory    654718 non-null  object 
 7   PropertyType        654718 non-null  object 
 8   IncGeo_BoroughName  654729 non-null  object 
 9   Easting_m           265087 non-null  float64
 10  Northing_m          265087 non-null  float64
 11  Easting_rounded     654729 non-null  int64  
 12  Northing_rounded    654729 non-null  int64  
 13  NumCalls            654714 non-null  float64
 14  source_file         654729 non-null  object 
dtypes: float64(3), int64(4), object(8)

In [ ]:
#NaN in Specail Service Type check
nan_count = inc_trunc[(inc_trunc['IncidentGroup'] == 'Special Service') & (inc_trunc['SpecialServiceType'].isna())].shape[0]

print(f"Count for 'Special Service' Incidents without SpecialServiceType: {nan_count}")

Count for 'Special Service' Incidents without SpecialServiceType: 170


In [52]:
#NaN in Specail Service Type check
nan_count = inc_trunc[(inc_trunc['IncidentGroup'] != 'Special Service') & (inc_trunc['SpecialServiceType'].isna())].shape[0]

print(f"Count for False Alarm or Fire Incidents without SpecialServiceType: {nan_count}")

Count for False Alarm or Fire Incidents without SpecialServiceType: 400431


In [53]:
inc_trunc.isna().sum()

IncidentNumber             0
DateOfCall                 0
CalYear                    0
HourOfCall                 0
IncidentGroup             54
SpecialServiceType    400601
PropertyCategory          11
PropertyType              11
IncGeo_BoroughName         0
Easting_m             389642
Northing_m            389642
Easting_rounded            0
Northing_rounded           0
NumCalls                  15
source_file                0
dtype: int64